# South African ASR on Kaggle

One notebook, several **stages**. Set `STAGE` in cell 1, then **Save Version → Save & Run All
(Commit)**. The run continues in the background for up to 12 hours with the browser closed.
Everything worth keeping is saved to private Hugging Face repos as it happens, so an interrupted
stage simply continues when you run it again.

| STAGE | What it does | Time on a T4 |
|---|---|---|
| `smoke` | tests + tiny end-to-end run (optional sanity check) | ~5 min |
| `baseline` | scores existing models on held-out test sets | ~4–6 h |
| `trial` | 500 training steps, isiZulu + isiXhosa | ~1 h |
| `train` | full training, all 11 languages (resumes if cut off) | ~8–12 h |
| `compare` | scores your trained model against the baseline | ~1–2 h |
| `report` | just shows the current results table | 1 min |

**One-time setup** (details in the README):
1. Kaggle account, **phone-verified** (needed for GPU and internet).
2. **Add-ons → Secrets → Add secret**: label `HF_TOKEN`, value = your Hugging Face **Write** token. Tick it
   for this notebook.
3. **Add Input** → your `sa-asr` dataset (the uploaded `sa-asr.zip`). If uploading is blocked, skip this;
   the notebook pulls the code from `<you>/sa-asr-code` on Hugging Face instead.
4. **Settings** (right panel): Accelerator **GPU T4 x2**, **Internet on**.

**Where to watch a background run:** your notebook → **Versions** → the running version → **Logs**,
or the files appearing in `huggingface.co/datasets/<you>/sa-asr-results`.

## 1. Settings: pick the stage here

In [ ]:
HF_USER = "KekeCode"      # your Hugging Face username
STAGE = "baseline"        # smoke | baseline | trial | train | compare | report

RESULTS_REPO = f"{HF_USER}/sa-asr-results"
MODEL_REPO = f"{HF_USER}/sa-asr-whisper-lora"
CODE_REPO = f"{HF_USER}/sa-asr-code"      # used only if no Kaggle dataset with the code is attached

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # the scripts use one GPU ("T4 x2" works; the second one idles)
import torch
assert torch.cuda.is_available(), "No GPU: Settings -> Accelerator -> GPU T4 x2 (needs a phone-verified account)"
gpu = torch.cuda.get_device_properties(0)
gpu_gb = gpu.total_memory / 2**30
if gpu_gb > 30:   # A100 / H100
    BATCH, EVAL_SAMPLES = "--batch-size 32 --grad-accum 1", 200
    BASELINE_MODELS = ["whisper-large-v3-turbo", "whisper-large-v3", "mms-1b-all", "dsfsi-anv-multilingual-turbo"]
else:             # T4 / P100: fewer test utterances, skip whisper-large-v3 (slowest, not the one we fine-tune)
    BATCH, EVAL_SAMPLES = "--batch-size 8 --grad-accum 4", 100
    BASELINE_MODELS = ["whisper-large-v3-turbo", "dsfsi-anv-multilingual-turbo", "mms-1b-all"]
BASELINE_ARG = " ".join(BASELINE_MODELS)
print(f"STAGE = {STAGE}")
print(f"GPU: {gpu.name} ({gpu_gb:.0f} GB); training batch {BATCH}; {EVAL_SAMPLES} test utterances per set")

## 2. Hugging Face login (token from Kaggle Secrets)

In [ ]:
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    raise RuntimeError("No HF_TOKEN secret: Add-ons -> Secrets -> add HF_TOKEN and tick it for this notebook") from e
os.environ["HF_TOKEN"] = token   # inherited by every script below
login(token=token)

## 3. Get the code
From the attached Kaggle dataset if there is one, otherwise from the private Hugging Face code repo.

In [ ]:
import glob, shutil, zipfile
WORK = "/kaggle/working/sa-asr"
found = sorted(glob.glob("/kaggle/input/**/pyproject.toml", recursive=True))
zips = sorted(glob.glob("/kaggle/input/**/sa-asr*.zip", recursive=True))
if found:
    shutil.copytree(os.path.dirname(found[0]), WORK, dirs_exist_ok=True)
    print("Code from Kaggle dataset:", os.path.dirname(found[0]))
elif zips:
    zipfile.ZipFile(zips[0]).extractall("/kaggle/working")
    print("Code from zip:", zips[0])
else:
    from huggingface_hub import snapshot_download
    snapshot_download(CODE_REPO, repo_type="dataset", local_dir=WORK)
    print("Code from Hugging Face:", CODE_REPO)
%cd {WORK}

## 4. Install and check access
Expect `User: …` and two OK lines.

In [ ]:
!pip install -q -e ".[dev]"
# Kaggle/Colab images may ship an old torchao that recent PEFT refuses to work with. We don't use it.
!pip uninstall -q -y torchao

!python -c "import sa_asr; from huggingface_hub import HfApi; print('User:', HfApi().whoami()['name'])"
!python -c "import sa_asr; from huggingface_hub import auth_check; auth_check('dsfsi-anv/za-african-next-voices', repo_type='dataset'); print('Swivuriso access: OK')" 2>/dev/null || echo "Swivuriso access: FAILED -> accept the terms at https://huggingface.co/datasets/dsfsi-anv/za-african-next-voices (as the same user). The run continues without Swivuriso."
!python -c "import sa_asr; from huggingface_hub import HfApi; HfApi().create_repo('{RESULTS_REPO}', repo_type='dataset', private=True, exist_ok=True); print('Write access: OK')" 2>/dev/null || echo "Write access: FAILED -> the HF_TOKEN secret needs Write permission, and HF_USER must be your username"

## 5. Run the stage

In [ ]:
import yaml
if STAGE == "compare":
    cfg = yaml.safe_load(open("configs/eval_baseline.yaml"))
    cfg["models"].append({"name": "sa-turbo-lora", "type": "whisper", "model": MODEL_REPO, "language_mode": "proxy"})
    yaml.safe_dump(cfg, open("configs/eval_compare.yaml", "w"), sort_keys=False)

STAGES = {
    "smoke": [
        "python -m pytest -q",
        "python scripts/evaluate.py --config configs/eval_smoke.yaml",
        "python scripts/train.py --config configs/train_smoke.yaml",
    ],
    "baseline": [
        f"python scripts/evaluate.py --config configs/eval_baseline.yaml --models {BASELINE_ARG} "
        f"--max-samples {EVAL_SAMPLES} --sync-repo {RESULTS_REPO}",
    ],
    "trial": [
        f"python scripts/train.py --config configs/train_lora.yaml {BATCH} --max-steps 500 --langs zul xho "
        f"--output-dir outputs/trial --hub-repo {HF_USER}/sa-asr-trial --resume",
    ],
    "train": [
        f"python scripts/train.py --config configs/train_lora.yaml {BATCH} --hub-repo {MODEL_REPO} --resume",
    ],
    "compare": [
        f"python scripts/evaluate.py --config configs/eval_compare.yaml --models sa-turbo-lora "
        f"--max-samples {EVAL_SAMPLES} --sync-repo {RESULTS_REPO}",
    ],
    "report": [
        f"python scripts/hub_sync.py pull results/baseline --repo {RESULTS_REPO}",
    ],
}
if STAGE not in STAGES:
    raise ValueError(f"STAGE must be one of {list(STAGES)}")

ip = get_ipython()
for cmd in STAGES[STAGE]:
    print(">>>", cmd, flush=True)
    ip.system(cmd)
    if ip.user_ns.get("_exit_code", 0) != 0:
        raise RuntimeError(f"Step failed (exit code {ip.user_ns['_exit_code']}): {cmd}")
print(f"Stage '{STAGE}' finished.")

## 6. Results table

In [ ]:
from IPython.display import Markdown, display
path = "results/baseline/summary.md"
if os.path.exists(path):
    display(Markdown(open(path, encoding="utf-8").read()))
else:
    print("No results yet: run the baseline stage (or the report stage to fetch saved results).")